In [2]:
import pandas as pd
import numpy as np
import string
from nltk.corpus import stopwords
from nltk import word_tokenize

In [4]:
import nltk 
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
df= pd.read_csv('emails.csv')
df.head()

,text,spam
0,Subject: Win a free iPhone now click here,1
1,Subject: Meeting tomorrow at 10am in the confe...,0
2,Subject: Congratulations you won a cash prize ...,1
3,Subject: Project report deadline is Friday ple...,0
4,Subject: Cheap viagra pills buy now limited offer,1


In [7]:
print(len(df))
print(df.spam.mean())

30
0.5


In [13]:
def preprocess_emails(df):
    df= df.sample(frac=1, ignore_index= True,random_state=42)
    X= df.text.apply(lambda X: X[9:]).to_numpy()
    Y= df.spam.to_numpy()
    return X,Y

In [15]:
X,Y = preprocess_emails(df)
print(X)
print(Y)

['Following up on our conversation yesterday'
 'Happy birthday wishing you all the best'
 'Feedback on the new product design'
 'Team building event next month RSVP' 'Claim your lottery winnings today'
 'Reminder dentist appointment on Monday'
 'Winner winner claim your prize immediately'
 "Act now limited stock available don't miss out"
 'Earn thousands from home no experience needed'
 'Win a free iPhone now click here'
 'Cheap viagra pills buy now limited offer'
 'Urgent your account will be suspended verify now'
 'Lunch with the team next Tuesday'
 'Please find attached the invoice for March'
 'Quarterly budget review meeting notes'
 'You have been selected for a special reward'
 'Meeting tomorrow at 10am in the conference room'
 'Congratulations you won a cash prize claim now'
 'Invitation to the annual company picnic'
 'Project report deadline is Friday please review'
 'Reminder to submit your timesheet by Friday'
 'Low interest loans approved instantly apply now'
 'Get rich quick

In [25]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
def preprocess_text(X):
    stop = set(stopwords.words('english') + list(string.punctuation))
    
    if isinstance(X, str):
        X = np.array([X])
    
    X_preprocessed = []
    for email in X:
        tokens = [w.lower() for w in word_tokenize(email) if w.lower() not in stop]
        X_preprocessed.append(np.array(tokens))
    
    if len(X_preprocessed) == 1:
        return X_preprocessed[0]
    return X_preprocessed

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
X_treated= preprocess_text(X)
print(X_treated)

[array(['following', 'conversation', 'yesterday'], dtype='<U12'), array(['happy', 'birthday', 'wishing', 'best'], dtype='<U8'), array(['feedback', 'new', 'product', 'design'], dtype='<U8'), array(['team', 'building', 'event', 'next', 'month', 'rsvp'], dtype='<U8'), array(['claim', 'lottery', 'winnings', 'today'], dtype='<U8'), array(['reminder', 'dentist', 'appointment', 'monday'], dtype='<U11'), array(['winner', 'winner', 'claim', 'prize', 'immediately'], dtype='<U11'), array(['act', 'limited', 'stock', 'available', "n't", 'miss'], dtype='<U9'), array(['earn', 'thousands', 'home', 'experience', 'needed'], dtype='<U10'), array(['win', 'free', 'iphone', 'click'], dtype='<U6'), array(['cheap', 'viagra', 'pills', 'buy', 'limited', 'offer'], dtype='<U7'), array(['urgent', 'account', 'suspended', 'verify'], dtype='<U9'), array(['lunch', 'team', 'next', 'tuesday'], dtype='<U7'), array(['please', 'find', 'attached', 'invoice', 'march'], dtype='<U8'), array(['quarterly', 'budget', 'review', 'm

In [44]:
TRAIN_SIZE = int(0.80 * len(X_treated))
X_train = X_treated[:TRAIN_SIZE]
Y_train = Y[:TRAIN_SIZE]
X_test  = X_treated[TRAIN_SIZE:]
Y_test  = Y[TRAIN_SIZE:]

In [45]:
print(len(X_train))
print(len(X_test))

24
6


In [46]:
def get_word_frequency(X, Y):
    word_dict = {}
    for email, cls in zip(X, Y):
        for word in set(email):
            if word not in word_dict:
                word_dict[word] = {"spam": 0, "ham": 0}
            if cls == 1:
                word_dict[word]["spam"] += 1
            else:
                word_dict[word]["ham"] += 1
    return word_dict

In [47]:
word_frequency= get_word_frequency(X_train,Y_train)

In [50]:
def get_class_frequency(Y):
    return {"spam": int(np.sum(Y)),"ham": int(len(Y)-np.sum(Y))}

In [51]:
class_frequency= get_class_frequency(Y_train)
print(class_frequency)

{'spam': 11, 'ham': 13}


In [55]:
def prob_word_given_class( word, cls, word_frequency, class_frequency):
    return word_frequency[word][cls] / class_frequency[cls]

In [58]:
print( prob_word_given_class("free","spam",word_frequency, class_frequency))

0.09090909090909091


In [59]:
print( prob_word_given_class("meeting","ham",word_frequency, class_frequency))

0.15384615384615385


In [65]:
def prob_email_given_class(treated_email, cls, word_frequency, class_frequency):
    prob = 1
    for word in treated_email:
        if word in word_frequency:
            prob *= prob_word_given_class(word, cls, word_frequency, class_frequency)
    return prob

In [74]:
email = ["free", "win", "meeting"]
print(prob_email_given_class(email,"ham", word_frequency, class_frequency))

0.0


In [77]:
def naive_bayes(treated_email, word_frequency, class_frequency, return_likelihood=False):
    prob_email_given_spam = prob_email_given_class(treated_email, "spam", word_frequency, class_frequency)
    prob_email_given_ham  = prob_email_given_class(treated_email, "ham",  word_frequency, class_frequency)
    
    total = class_frequency["spam"] + class_frequency["ham"]
    p_spam = class_frequency["spam"] / total
    p_ham  = class_frequency["ham"]  / total
    
    spam_likelihood = p_spam * prob_email_given_spam
    ham_likelihood  = p_ham  * prob_email_given_ham
    
    if return_likelihood:
        return (spam_likelihood, ham_likelihood)
    return 1 if spam_likelihood >= ham_likelihood else 0

In [78]:
nouveaux_emails = [
    "Subject: Free iPhone click here now",
    "Subject: Meeting at 10am tomorrow",
    "Subject: Claim your cash prize immediately",
    "Subject: Please review the report by Friday",
    "Subject: Cheap loans approved instantly",
    "Subject: Lunch with the team next week",
]

for email in nouveaux_emails:
    treated = preprocess_text(email)
    prediction = naive_bayes(treated, word_frequency, class_frequency)
    spam_lik, ham_lik = naive_bayes(treated, word_frequency, class_frequency, return_likelihood=True)
    label = "🚨 SPAM" if prediction == 1 else "✅ HAM"
    print(f"{label}  | spam={spam_lik:.2e}  ham={ham_lik:.2e}  | {email}")

🚨 SPAM  | spam=3.44e-04  ham=0.00e+00  | Subject: Free iPhone click here now
✅ HAM  | spam=0.00e+00  ham=4.93e-04  | Subject: Meeting at 10am tomorrow
🚨 SPAM  | spam=1.88e-04  ham=0.00e+00  | Subject: Claim your cash prize immediately
✅ HAM  | spam=0.00e+00  ham=3.41e-04  | Subject: Please review the report by Friday
🚨 SPAM  | spam=3.13e-05  ham=0.00e+00  | Subject: Cheap loans approved instantly
✅ HAM  | spam=0.00e+00  ham=9.86e-04  | Subject: Lunch with the team next week


In [79]:
def predict(X, word_frequency, class_frequency):
    return np.array([naive_bayes(email, word_frequency, class_frequency) for email in X])

Y_pred = predict(X_test, word_frequency, class_frequency)

# Matrice de confusion
TP = np.sum((Y_pred == 1) & (Y_test == 1))
TN = np.sum((Y_pred == 0) & (Y_test == 0))
FP = np.sum((Y_pred == 1) & (Y_test == 0))
FN = np.sum((Y_pred == 0) & (Y_test == 1))

accuracy  = (TP + TN) / len(Y_test)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Accuracy  : {accuracy:.2%}")
print(f"Precision : {precision:.2%}")
print(f"Recall    : {recall:.2%}")
print(f"F1-score  : {f1:.2%}")
print(f"\nMatrice de confusion :")
print(f"  TP={TP}  FP={FP}")
print(f"  FN={FN}  TN={TN}")

Accuracy  : 66.67%
Precision : 100.00%
Recall    : 50.00%
F1-score  : 66.67%

Matrice de confusion :
  TP=2  FP=0
  FN=2  TN=2
